# AI Health Report Analyzer

Step 1: Install Required Libraries

In [1]:
# Install required packages

!pip install streamlit
!pip install pytesseract
!pip install pdf2image
!pip install google-generativeai
!pip install pillow
!pip install opencv-python
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 71.0 MB/s eta 0:00:00


Step 2: Upload Medical Report (PDF/Image)

In [2]:
from google.colab import files

# Upload medical report
uploaded = files.upload()

# Show uploaded file names
for file_name in uploaded.keys():
    print("Uploaded file:", file_name)

Saving blood-test-report.pdf to blood-test-report.pdf
Uploaded file: blood-test-report.pdf


Step 2.1: Check File Type

In [3]:
import os

file_path = list(uploaded.keys())[0]

extension = os.path.splitext(file_path)[1].lower()

print("File Name:", file_path)
print("File Type:", extension)

File Name: blood-test-report.pdf
File Type: .pdf


Step 3: Install Tesseract OCR Engine

In [4]:
!apt-get install tesseract-ocr -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [5]:
#Check installation:
import pytesseract

print(pytesseract.get_tesseract_version())

Step 4: Convert PDF to Image

In [8]:
import os

!apt-get update
!apt install poppler-utils -y

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 Packages [38.6 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,228 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Pac

In [9]:
from pdf2image import convert_from_path

if extension == ".pdf":
    pages = convert_from_path(file_path)

    images = []

    for i, page in enumerate(pages):
        image_name = f"page_{i}.jpg"
        page.save(image_name, "JPEG")
        images.append(image_name)

    print("PDF converted successfully")

PDF converted successfully


Step 5: Handle the images (if they are in JPG/PNG format)

In [10]:
if extension in [".jpg", ".jpeg", ".png"]:
    images = [file_path]

print(images)

['page_0.jpg']


Step 6: OCR – Extract text from images

In [11]:
from PIL import Image
import pytesseract

medical_text = ""

for img in images:
    image = Image.open(img)

    text = pytesseract.image_to_string(image)

    medical_text += text + "\n"


print("Extracted Medical Text:")
print(medical_text)

Extracted Medical Text:
Blood Test Report Collection: March 26, 2026

Sample: Venous Blood
Laboratory Diagnostics Report

 

Patient Information

 

 

Patient: Jane Doe Patient ID: PT-00412
Date of Birth: March 15, 1985 Ordering Physician: Dr. Sarah Chen

Test Results

 

 

TEST NAME RESULT UNIT REFERENCE RANGE FLAG
Hemoglobin 14.2 g/dL 12.0-17.5

White Blood Cells 7.8 x10%3/uL 4.5-11.0

Platelets 245 x10*3/uL 150-400

Glucose (Fasting) 112 mg/dL 70-100 HIGH
Creatinine 0.9 mg/dL 0.7-1.3

 

Physician Notes

 

 

Glucose slightly elevated. Recommend follow-up fasting glucose in 3 months.

 

 

Lab Technician: M. Rodriguez
Results should be interpreted by a qualified healthcare provider.




Step 7: Clean the extracted text

In [12]:
import re

clean_text = re.sub(r"\s+", " ", medical_text)

print(clean_text)

Blood Test Report Collection: March 26, 2026 Sample: Venous Blood Laboratory Diagnostics Report Patient Information Patient: Jane Doe Patient ID: PT-00412 Date of Birth: March 15, 1985 Ordering Physician: Dr. Sarah Chen Test Results TEST NAME RESULT UNIT REFERENCE RANGE FLAG Hemoglobin 14.2 g/dL 12.0-17.5 White Blood Cells 7.8 x10%3/uL 4.5-11.0 Platelets 245 x10*3/uL 150-400 Glucose (Fasting) 112 mg/dL 70-100 HIGH Creatinine 0.9 mg/dL 0.7-1.3 Physician Notes Glucose slightly elevated. Recommend follow-up fasting glucose in 3 months. Lab Technician: M. Rodriguez Results should be interpreted by a qualified healthcare provider. 


Step 8: Abnormal Value Detection (Basic AI Logic)

In [18]:
import re

results = []

text = clean_text.lower()

# 🔹 Hemoglobin (flexible)
hb_match = re.search(r"hemoglobin\s+([\d.]+)", text)

if hb_match:
    hb = float(hb_match.group(1))
    if hb < 12:
        results.append("Low Hemoglobin detected")
    elif hb > 17.5:
        results.append("High Hemoglobin detected")

# 🔹 Glucose (handles fasting/random)
glucose_match = re.search(r"glucose.*?([\d.]+)", text)

if glucose_match:
    glucose = float(glucose_match.group(1))
    if glucose > 100:
        results.append("High Glucose detected")

# 🔹 WBC
wbc_match = re.search(r"white blood cells\s+([\d.]+)", text)

if wbc_match:
    wbc = float(wbc_match.group(1))
    if wbc > 11:
        results.append("High WBC detected")

# 🔹 Platelets
platelets_match = re.search(r"platelets\s+([\d.]+)", text)

if platelets_match:
    platelets = float(platelets_match.group(1))
    if platelets < 150:
        results.append("Low Platelets detected")

print(results)

['High Glucose detected']


Step 9: Doctor Recommendation Automation

In [19]:
doctors = []

for issue in results:

    if "Glucose" in issue:
        doctors.append("Endocrinologist")

    if "Hemoglobin" in issue:
        doctors.append("General Physician")


print("Recommended Doctors:")
print(set(doctors))

Recommended Doctors:
{'Endocrinologist'}


Step 10: Save Report History in Database

In [20]:
import sqlite3

conn = sqlite3.connect("medical_records.db")

cursor = conn.cursor()


cursor.execute("""
CREATE TABLE IF NOT EXISTS reports(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    report TEXT,
    issues TEXT
)
""")


cursor.execute(
    "INSERT INTO reports(report, issues) VALUES (?, ?)",
    (clean_text, str(results))
)


conn.commit()

print("Data saved successfully")

Data saved successfully


Step 11: View Previous Reports

In [21]:
for row in cursor.execute(
    "SELECT * FROM reports"
):
    print(row)

(1, 'Blood Test Report Collection: March 26, 2026 Sample: Venous Blood Laboratory Diagnostics Report Patient Information Patient: Jane Doe Patient ID: PT-00412 Date of Birth: March 15, 1985 Ordering Physician: Dr. Sarah Chen Test Results TEST NAME RESULT UNIT REFERENCE RANGE FLAG Hemoglobin 14.2 g/dL 12.0-17.5 White Blood Cells 7.8 x10%3/uL 4.5-11.0 Platelets 245 x10*3/uL 150-400 Glucose (Fasting) 112 mg/dL 70-100 HIGH Creatinine 0.9 mg/dL 0.7-1.3 Physician Notes Glucose slightly elevated. Recommend follow-up fasting glucose in 3 months. Lab Technician: M. Rodriguez Results should be interpreted by a qualified healthcare provider. ', "['High Glucose detected']")


Step 12: Gemini Generative AI Integration (AI Summary)

In [35]:
import google.generativeai as genai

genai.configure(api_key="AQ.Ab8RN6IYpd-_WUeFxKLUBOZ3K-k8LhgqbRtNXydrKPos2EL0fg")

model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content("Hello")
print(response.text)

Hello! How can I help you today?


In [36]:
import google.generativeai as genai

genai.configure(api_key="YOUR_GEMINI_API_KEY")

model = genai.GenerativeModel("gemini-2.5-flash")

prompt = f"""
You are an AI medical assistant.

Analyze the following medical report and provide:

1. Simple summary
2. Abnormal findings
3. Health concerns
4. Recommended specialist

Do not provide a final diagnosis.

Medical Report:
{clean_text}
"""

response = model.generate_content(prompt)

print(response.text)

Here's an analysis of the provided medical report:

1.  **Simple summary**
    Jane Doe's recent blood test results show that most parameters are within normal limits. However, her fasting glucose level was slightly elevated, which the ordering physician has noted and recommended a follow-up test for in three months.

2.  **Abnormal findings**
    *   **Glucose (Fasting):** Result: 112 mg/dL (Reference Range: 70-100 mg/dL) - Elevated (HIGH).

3.  **Health concerns**
    An elevated fasting glucose level, as seen in this report, can indicate a potential increased risk for impaired glucose tolerance, prediabetes, or type 2 diabetes. It suggests that the body may not be processing blood sugar as effectively as it should.

4.  **Recommended specialist**
    A **primary care physician** is appropriate for initial follow-up, management, and repeat testing as suggested. If the elevated glucose persists or worsens, a referral to an **endocrinologist** might be considered for specialized evalua